In [4]:
!pip install -q streamlit pyngrok pandas numpy requests scikit-learn plotly


In [7]:
%%writefile gdp_predictor.py


import numpy as np
import pandas as pd
import requests
import streamlit as st
import plotly.graph_objects as go

WB_BASE = "https://api.worldbank.org/v2"
GDP_CODE = "NY.GDP.MKTP.CD"  # GDP (current US$)


#we can alter these indicators
INDICATORS = {
    "C": "NE.CON.PRVT.CD",     # Household final consumption expenditure (current US$)
    "I": "NE.GDI.TOTL.CD",     # Gross capital formation (current US$)
    "G": "NE.CON.GOVT.CD",     # General government final consumption expenditure (current US$)
    "X": "NE.EXP.GNFS.CD",     # Exports of goods and services (current US$)
    "M": "NE.IMP.GNFS.CD",     # Imports of goods and services (current US$)
    "CPI_INFL": "FP.CPI.TOTL.ZG",  # Inflation, consumer prices (annual %)
    "UNEMP": "SL.UEM.TOTL.ZS",     # Unemployment, total (% of total labor force)
}

st.set_page_config(page_title="GDP Predictor", page_icon="📈", layout="wide")
st.title("GDP Predictor")

@st.cache_data(show_spinner=False, ttl=24*3600)
def fetch_countries():
    url = f"{WB_BASE}/country?format=json&per_page=400"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    j = r.json()
    df = pd.json_normalize(j[1])
    df = df[df["region.id"].ne("NA")]
    return df[["id", "name"]].sort_values("name").reset_index(drop=True)

@st.cache_data(show_spinner=False, ttl=6*3600)
def fetch_gdp(country):
    url = f"{WB_BASE}/country/{country}/indicator/{GDP_CODE}?format=json&per_page=20000"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    j = r.json()
    if not isinstance(j, list) or len(j) < 2 or j[1] is None:
        return pd.DataFrame(columns=["year", "GDP"])
    df = pd.json_normalize(j[1])[["date", "value"]]
    df.columns = ["year", "GDP"]
    df["year"] = pd.to_numeric(df["year"], errors="coerce")
    df["GDP"] = pd.to_numeric(df["GDP"], errors="coerce")
    return df.dropna(subset=["year", "GDP"]).sort_values("year").reset_index(drop=True)

def forecast_gdp(hist, horizon):
    df = hist[["year", "GDP"]].copy()
    df = df[df["GDP"] > 0]
    df["log_gdp"] = np.log(df["GDP"])

    X = df["year"].values.reshape(-1, 1)
    y = df["log_gdp"].values

    coeffs = np.polyfit(X.flatten(), y, 1)
    future_years = np.array(range(df["year"].max() + 1, df["year"].max() + horizon + 1)).reshape(-1, 1)

    future_log = np.polyval(coeffs, future_years.flatten())
    future_gdp = np.exp(future_log)

    fc_df = pd.DataFrame({
        "year": future_years.flatten(),
        "GDP": future_gdp
    })
    return fc_df

with st.sidebar:
    st.header("Controls")
    countries = fetch_countries()
    options = dict(zip(countries["name"], countries["id"]))
    default_name = "United States" if "United States" in options else list(options.keys())[0]
    country_name = st.selectbox("Country", list(options.keys()),
                                index=list(options.keys()).index(default_name))
    country_code = options[country_name]
    horizon = st.slider("Forecast horizon (years)", 5, 20, 10, 1)

with st.spinner("Fetching data..."):
    hist = fetch_gdp(country_code)

if hist.empty:
    st.warning("No GDP data available for this country.")
    st.stop()

fcst = forecast_gdp(hist, horizon)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=hist["year"], y=hist["GDP"], mode="lines+markers", name="Historical GDP",
    hovertemplate="Year %{x}<br>GDP $%{y:,.0f}<extra></extra>",
    line=dict(width=2, color="#1f77b4"), marker=dict(size=6, color="#1f77b4")
))
fig.add_trace(go.Scatter(
    x=fcst["year"], y=fcst["GDP"], mode="lines+markers", name="Forecast",
    hovertemplate="Year %{x}<br>GDP $%{y:,.0f}<extra></extra>",
    line=dict(width=2, color="#ff7f0e"), marker=dict(size=6, color="#ff7f0e")
))
fig.update_layout(
    title=f"GDP (current US$): {country_name}",
    xaxis_title="Year", yaxis_title="GDP (US$)",
    hovermode="x unified",
    template="plotly_white"
)

st.plotly_chart(fig, use_container_width=True)

Overwriting gdp_predictor.py


In [8]:
from pyngrok import ngrok

ngrok.set_auth_token("342yzWfv1WrWG97Kw6yZUxPDxQD_6rNJ8h9c3KRX71h5Tzxyh")

public_url = ngrok.connect(8501, "http").public_url
print("🔗  Your public Streamlit app URL:", public_url)


!STREAMLIT_SERVER_HEADLESS=true streamlit run gdp_predictor.py --server.port 8501 --server.address 0.0.0.0


🔗  Your public Streamlit app URL: https://nonallegorical-immanuel-palpably.ngrok-free.dev



  You can now view your Streamlit app in your browser.

  URL: http://0.0.0.0:8501

  Stopping...
  Stopping...
